<a href="https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w5_reranking/llm_260410_reranking_threshold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 260410 Score Filtering & LLM Judge 심화

**5주차 Day 4** - Reranking 후 점수 필터링 + LLM 평가 편향 탐지 & 앱상블

어제까지 reranking 방법들(keyword, BM25, LLM, listwise)을 배웠다.  
오늘은 reranking 결과에서 **나쁜 문서를 걸러내는 방법(Score Filtering)**과,  
LLM으로 답변 품질을 평가할 때의 **편향(bias) 탐지**와 **앱상블 평가**를 다룬다.

| 주제 | 비유 |
|------|------|
| Fixed Threshold | 시험 60점 미만 불합격 -- 절대적 기준선 |
| Dynamic Threshold | 학급 평균-표준편차 기준으로 자르는 상대적 기준선 |
| Score Gap | 점수 배열에서 가장 큰 떨어짐이 있는 지점에서 자르는 것 |
| Adaptive Filter | 점수 분포를 보고 위 3개 전략 중 자동 선택 |
| 길이 편향 | LLM이 긴 답변에 후하게 점수를 주는 경향 |
| 앱상블 Judge | 여러 심사위원의 점수를 평균/절삭/가중평균으로 합치는 것 |

## 0. Setup

In [ ]:
# Colab 환경이면 아래 주석 해제
# !pip install -q openai langchain langchain-openai langchain-community langchain-classic faiss-cpu python-dotenv

In [ ]:
# --- Colab Secrets 사용 시 ---
# import os
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# --- 로컬 .env 사용 시 ---
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from dotenv import load_dotenv

load_dotenv()

MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. 실습 데이터 준비

어제(260409)와 동일한 8개 문서로 시작한다.

In [ ]:
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성 기법으로 외부 지식을 LLM에 결합하여 할루시네이션을 줄입니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.", metadata={"id": "d5"}),
    Document(page_content="파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.", metadata={"id": "d6"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d7"}),
    Document(page_content="토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. BPE, WordPiece 등이 사용됩니다.", metadata={"id": "d8"}),
]

vectorstore = FAISS.from_documents(documents, embeddings_model)
bm25_retriever = BM25Retriever.from_documents(documents, k=5)

In [ ]:
# 문서별 임베딩 캐시 (나중에 코사인 유사도 계산용)
doc_embeddings = {}

def get_embedding(text):
    return np.array(embeddings_model.embed_query(text))

for doc in documents:
    doc_embeddings[doc.metadata['id']] = get_embedding(doc.page_content)

In [ ]:
# 기본 벡터 검색 함수 (어제와 동일)
def vector_search(query, vectorstore, top_k=5):
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    return [(doc, 1.0 / (1.0 + score)) for doc, score in results]

query = "트랜스포머와 BERT의 관계"
results = vector_search(query, vectorstore)
results

## 2. Listwise Reranking

어제는 **pointwise**(1개씩 평가)와 **pairwise**(2개씩 비교)를 배웠다.  
오늘은 **listwise** -- 문서 전체를 한 번에 LLM에 넘기고 순위를 정하는 방법이다.

| 방법 | 비유 | 복잡도 |
|------|------|----------|
| Pointwise | 시험지 하나씩 채점 | O(N) |
| Pairwise | 두 명씩 비교해서 순위 | O(N²) |
| Listwise | 전체 시험지를 한 번에 순위 매기기 | O(1) -- 단 한 번의 LLM 호출 |

In [ ]:
def llm_listwise_rerank(query, search_results):
    """LLM에게 문서 전체를 보여주고 관련성 순서로 정렬시키는 방식"""
    docs_text = '\n'.join(
            f"{i+1}. [{doc.metadata['id']}] {doc.page_content[:80]}" for i, (doc, _) in enumerate(search_results)
    )
    
    ranking_chain = ChatPromptTemplate.from_messages([
        ('system', "당신은 문서 랭킹 시스템입니다"),
        ('human', """다음 문서들을 쿼리와의 관련성 순서로 정렬하세요.
        
        쿼리 : {query}
        
        문서들 : {docs_text}
        
        가장 관련성 높은 순서대로 문서 번호를 쉴표로 나열하세요 (예: 3,1,5,2,4):""")
    ]) | llm | StrOutputParser()
    
    result = ranking_chain.invoke({'query' : query, 'docs_text' : docs_text})
    
    # 결과 파싱: "3,1,5,2,4" -> [3,1,5,2,4]
    order = [int(x.strip()) for x in result.strip().split(',')]
    
    # 순위에 따라 점수 부여 (1등=1.0, 2등=0.9, ...)
    docs_list = [doc for doc, _ in search_results]
    reranked = []
    for rank, idx in enumerate(order):
        if 1<=idx<=len(docs_list):
            docs = docs_list[idx-1]
            reranked.append((docs, 1.0-rank*0.1))
            
    return reranked

In [ ]:
listwise = llm_listwise_rerank(query, results)
listwise

## 3. Score Filtering -- 나쁜 문서 걸러내기

Reranking 후에도 점수가 낮은 문서가 포함될 수 있다.  
Generator에 넓으면 노이즈가 되니까, **일정 기준 이하는 잘라내자**.  

비유: 레스토랑에서 주문받은 요리 5개 중, 품질 미달 요리는 손님한테 내보내지 않는 것.

```
RAG 파이프라인:
Retriever -> Reranker -> [Score Filter] -> Generator
                          ^여기서 나쁜 문서 제거
```

### 3가지 필터링 전략

| 전략 | 설명 | 장점 | 단점 |
|------|------|------|------|
| Fixed Threshold | 0.5 이하 무조건 제거 | 단순하고 직관적 | 데이터 분포 무시 |
| Dynamic Threshold | 평균 - k*표준편차 기준 | 데이터 분포 반영 | k값 튜닝 필요 |
| Score Gap | 점수 간 가장 큰 간격에서 잘름 | 자연스러운 군집 분리 | 점수가 균일하면 실패 |

In [ ]:
class ScoreFilter:
    
    @staticmethod
    def fixed_threshold(scored_docs, threshold=0.5):
        """고정 임계값: threshold 이상만 통과
        비유: 시험 60점 미만 = 불합격"""
        return [(doc, s) for doc, s in scored_docs if s >= threshold]
    
    @staticmethod
    def dynamic_threshold(scored_docs, std_factor=1):
        """동적 임계값: 평균 - k*표준편차
        비유: 학급 평균이 80점이면 합격 기준도 올라가는 것"""
        scores = [s for _, s in scored_docs]
        threshold = np.mean(scores) - std_factor * np.std(scores)
        return [(doc, s) for doc, s in scored_docs if s >= threshold]
    
    @staticmethod
    def score_gap(scored_docs, min_docs=2):
        """점수 간격이 가장 큰 지점에서 잘름
        비유: [0.9, 0.85, 0.7, | 0.5, 0.2] <- 0.7과 0.5 사이 간격이 0.2로 가장 크니까 여기서 잘름"""
        if len(scored_docs) <= min_docs:
            return scored_docs
        
        scores = [s for _, s in scored_docs]
        # 인접한 점수 간의 차이를 계산
        gaps = [(scores[i] - scores[i+1], i+1) for i in range(len(scores)-1)]
        # 가장 큰 간격의 위치를 찾음
        max_gap_idx = max(gaps, key=lambda x:x[0])[1]
        cut = max(min_docs, max_gap_idx)  # 최소 min_docs개는 보장
        return scored_docs[:cut]

In [ ]:
# 테스트 데이터: 점수가 높은 것부터 낮은 것까지 5개
test_scores = [(Document(page_content="", metadata={'id': f'd{i}'}), s) for i, s in enumerate([0.9, 0.85, 0.7, 0.5, 0.2])]
test_scores

In [ ]:
# Fixed: 0.5 이상만 통과 -> d0(0.9), d1(0.85), d2(0.7), d3(0.5) = 4개
filtered = ScoreFilter.fixed_threshold(test_scores, threshold=0.5)
print(f"Fixed (>=0.5): {len(filtered)}개 통과")
filtered

In [ ]:
# Dynamic: 평균(0.63) - 1*표준편차(0.26) = 0.37 이상 -> 4개 통과
filtered = ScoreFilter.dynamic_threshold(test_scores, std_factor=1)
print(f"Dynamic (mean-1*std): {len(filtered)}개 통과")
filtered

In [ ]:
# Score Gap: [0.9, 0.85, 0.7, | 0.5, 0.2]
# 간격: 0.05, 0.15, 0.2, 0.3 -> 최대 간격 0.3은 idx=4위치이지만, 0.2간격이 idx=3에서...
filtered = ScoreFilter.score_gap(test_scores, min_docs=2)
print(f"Score Gap: {len(filtered)}개 통과")
filtered

### 3-1. AdaptiveFilter -- 점수 분포에 따라 자동 선택

점수 분포를 보고 3가지 전략 중 자동으로 골라주는 필터.  
비유: 식당에서 손님이 많으면 겄속 주문, 적으면 지금 있는 것만으로 돌리는 적응형 운영.

In [ ]:
class AdaptiveFilter:
    
    @staticmethod
    def adapt_filter(scored_docs):
        """점수 분포에 따라 필터링 전략 자동 선택"""
        scores = [s for _, s in scored_docs]
        std = np.std(scores)
        score_range = max(scores) - min(scores)
        
        if std > 0.2:
            # 편차가 크다 = 점수 차이가 확실하다 -> Score Gap으로 자르는 게 효과적
            result = ScoreFilter.score_gap(scored_docs, min_docs=2)
        elif score_range < 0.1:
            # 점수가 거의 비슷비슷하다 = 구분이 어렵다 -> 절반만 선택
            result = scored_docs[:len(scored_docs)//2]
        else:
            # 중간 -> Dynamic Threshold
            result = ScoreFilter.dynamic_threshold(scored_docs, std_factor=1)
        
        return result

In [ ]:
filtered = AdaptiveFilter.adapt_filter(test_scores)
print(f"Adaptive: {len(filtered)}개 통과")
filtered

## 4. LLM-as-Judge -- Pointwise 평가 (복습)

어제부터 시작한 LLM 평가(LLM-as-Judge).  
**핵심 아이디어**: LLM에게 프롬프트로 "이 답변 몇 점?"을 물어봤더니 평가를 해준다.  

오늘은 여기에 **루브릭(rubric)**을 추가해서 점수 기준을 명확하게 준다.

비유: 시험 채점할 때 점수 기준표 없이 채점하면 주관적 -> 루브릭이 있으면 누가 채점해도 비슷한 점수

In [ ]:
# 루브릭: 5점 만점 기준표
RUBRIC = {
    5: "정확하고 완전하며, 예시와 설명이 풍부하다",
    4: "정확하고 핵심을 다루지만, 일부 세부사항이 부족하다",
    3: "대체로 정확하지만, 중요한 내용 일부가 누락되었다",
    2: "부분적으로만 정확하거나, 핵심을 빗나갔다",
    1: "부정확하거나 질문과 무관하다",
}

def pointwise_judge(question, answer, rubric=RUBRIC):
    """루브릭 기반 Pointwise 평가 -- 1개 답변을 1~5점으로 채점"""
    rubric_text = '\n'.join(
        f' {k} 점: {v}' for k, v in sorted(rubric.items(), reverse=True)
    )
    
    prompt = f"""다음 답변을 아래 루브릭에 따라 평가해주세요.
    
    질문 : {question}
    답변 : {answer}
    
    루브릭 :
    {rubric_text}
    
    반드시 JSON으로 답하세요:
    {{"score": 1-5, "reasoning" : "이유"}}"""
    
    result = llm.invoke(prompt).content
    result = result.replace('```json', '').replace('```', '')
    return json.loads(result)

In [ ]:
# 테스트: 3개 답변의 점수 차이 확인
question = "REST API와 GraphQL의 차이점을 설명해주세요"
answers = [
    "REST는 리소스 단위 URL에 HTTP 메서드를 사용하고, GraphQL은 단일 엔드포인트에서 클라이언트가 필요한 데이터를 쿼리합니다. REST는 오버/언더 페칭 문제가 있고, GraphQL은 이를 해결하지만 캐싱이 어렵습니다.",
    "REST는 URL을 사용하고 GraphQL은 쿼리를 사용합니다.",
    "둘 다 API입니다.",
]

for i, ans in enumerate(answers):
    result = pointwise_judge(question, ans)
    print(f"답변 {i+1}: {ans[:30]}...")
    print(f"  점수: {result.get('score')}/5 - {result.get('reasoning', '')[:50]}")
    print()

## 5. CoT / Few-shot / Reference-based Judge

어제 배운 프롬프팅 기법들을 LLM Judge에 적용한다.  
전부 **O(N)** 복잡도 -- 답변 1개당 LLM 호출 1번.

| 방법 | 핵심 | 비유 |
|------|------|------|
| CoT Judge | "단계별로 분석하세요" 추가 | 채점할 때 풀이과정도 적으라고 요구 |
| Few-shot Judge | 평가 예시를 2개 보여줌 | 선배 채점 결과를 참고자료로 제공 |
| Reference Judge | 정답을 기준으로 비교 | 모범답안 대비 얼마나 맞는지 |

In [ ]:
def cot_judge(question, answer):
    """단계별 분석(Chain of Thought) 기반 평가
    비유: 채점할 때 '어떻게 이 점수를 줬는지' 풀이과정을 써라고 요구"""
    prompt = f"""다음 답변을 단계별로 분석하세요.
    
    질문 : {question}
    답변 : {answer}
    
    step1: 질문이 요구하는 핵심 포인트를 나열하세요
    step2: 답변이 각 포인트를 다루는지 확인하세요
    step3: 답변의 강점과 약점을 정리하세요
    step4: 종합점수(1~5)를 부여하세요
    
    최종 JSON:
    {{"steps": ["step1 결과", "step2 결과", "step3 결과"], "score":1~5}}"""
    
    result = llm.invoke(prompt).content
    cleaned = result.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1].replace("json", "")
    return json.loads(cleaned)

In [ ]:
def fewshot_judge(question, answer):
    """예시 기반 평가 -- 높은 점수/낮은 점수 예시를 보여주고 평가시킴
    비유: 선배 리포트 샘플을 보여주고 '이런 수준으로 써줘' 하는 것"""
    prompt = f"""다음 답변을 1~5점으로 평가하세요.
    
    ===평가예시===
    
    질문: 파이썬 리스트란?
    답변: 파이썬 리스트는 순서가 있는 변경 가능한 컴렉션입니다. []로 생성하며 다양한 타입을 저장할 수 있습니다.
    점수: 4
    이유: 핵심 특성(순서, 변경가능)을 정확히 설명. 메서드 예시가 있었으면 5점.

    질문: 변수란 무엇인가요?
    답변: 값을 저장하는 것
    점수: 2
    이유: 맞지만 너무 짧고 구체적 설명이 부족.
    
    ===실제평가===
    
    질문 : {question}
    답변 : {answer}
    
    JSON으로 답하세요: {{"score": 1~5, "reasoning": "이유"}}"""
    
    result = llm.invoke(prompt).content
    cleaned = result.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1].replace("json", "")
    return json.loads(cleaned)

In [ ]:
def reference_based_judge(question, reference, answer):
    """정답(모범답안) 기반 평가
    비유: 모범답안과 대조해서 학생 답안 채점"""
    prompt = f"""당신은 AI 교육 전문가입니다.
    정답을 기준으로 학생의 답변을 평가하세요.
    
    질문 : {question}
    정답 : {reference}
    학생 답변 : {answer}
    
    평가 기준:
    - 정답의 핵심 포인트를 얼마나 포함하는가
    - 사실적으로 틀린 내용이 있는가
    - 정답에 없는 올바른 추가 정보가 있는가
    
    JSON으로 답하세요:
    {{"score": 1-5, "covered_points" : ["포함된 핵심 포인트들"], "missing_points":["누락된 포인트들"], "errors" : ["틀린 내용"]}}"""
    
    result = llm.invoke(prompt).content
    cleaned = result.replace('```json', '').replace('```', '')
    return json.loads(cleaned)

In [ ]:
# CoT Judge 테스트: 4대 원칙 중 하나가 누락된 답변
question = "객체지향 프로그래밍의 4대 원칙을 설명해주세요"
answer = "측술화, 상속, 다형성이 있습니다. 측술화는 데이터를 숨기는 것이고 상속은 부모 클래스를 물려받는 것입니다."

cot = cot_judge(question, answer)
print(f"점수: {cot.get('score')}/5")
for i, step in enumerate(cot.get('steps', [])):
    print(f"  step{i+1}: {step[:60]}")

In [ ]:
# Few-shot Judge 테스트
question = "Git이란 무엇인가요?"
answer = "코드 버전 관리 도구입니다."
fewshot_judge(question, answer)

In [ ]:
# Reference-based Judge 테스트
question = "TCP와 UDP의 차이점은?"
reference = "TCP는 연결지향적이고 신뢰성 있는 전송을 보장하며, UDP는 비연결지향적이고 빠르지만 신뢰성을 보장하지 않습니다."
answer = "TCP는 느리지만 안정적이고 UDP는 빠르지만 데이터 손실 가능성이 있습니다."

result = reference_based_judge(question, reference, answer)
print(f"점수: {result.get('score')}/5")
print(f"포함된 포인트: {result.get('covered_points')}")
print(f"누락된 포인트: {result.get('missing_points')}")

## 6. 다차원 평가 (Multi-Dimensional Evaluation)

지금까지는 "몇 점?"으로 하나의 점수만 매겼다.  
다차원 평가는 **여러 측면**에서 독립적으로 점수를 매기는 것.

비유: 영화 리뷰에서 "종합점수 7점"보다 "스토리 9점, 영상 8점, 음악 6점, 연기 7점"이 더 유용한 정보.

In [ ]:
# 평가 차원 정의
EVAL_DIMENSIONS = {
    "accuracy": "사실적으로 정확한 정보만 포함하는가 (1-5)",
    "relevance": "질문의 의도와 범위에 적합한 답변인가 (1-5)",
    "completeness": "핵심 포인트를 빠짐없이 다루는가 (1-5)",
    "coherence": "논리적 흐름과 구조가 일관적인가 (1-5)",
    "practicality": "답변에 실제 사용 가능한 코드나 구체적인 방법이 포함되어 있는가 (1-5)"
}

In [ ]:
def multidim_judge(question, answer, dimensions=EVAL_DIMENSIONS):
    """다차원 평가: 각 차원을 독립적으로 점수화"""
    dim_text = '\n'.join(f" - {k}: {v}" for k, v in dimensions.items())
    dim_keys = list(dimensions.keys())
    
    prompt = f"""다음 답변을 아래 차원별로 독립적으로 평가하세요.
    각 차원은 다른 차원에 영향을 받지 않고 독립 평가합니다.
    
    질문 : {question}
    답변 : {answer}
    
    평가 차원:
    {dim_text}
    
    JSON으로 답하세요:
    {{"scores": {{{','.join(f'"{k}": 1-5' for k in dim_keys)}}}, "summary": "한 줄 요약"}}"""
    
    result = llm.invoke(prompt).content
    cleaned = result.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1].replace("json", "")
    return json.loads(cleaned)

In [ ]:
# 테스트: 완전한 답변 vs 불완전한 답변
question = "머신러닝과 딥러닝의 차이점을 설명해주세요"
answers = [
    ("머신러닝은 데이터에서 패턴을 학습하는 넓은 개념이고, 딥러닝은 그 중 다층 신경망을 사용하는 방법입니다. 머신러닝에는 SVM, 결정트리 등이 있고, 딥러닝에는 CNN, RNN 등이 있습니다.", "완전한 답변"),
    ("둘 다 AI 기술입니다. 딥러닝이 더 발전된 기술입니다.", "불완전한 답변"),
]

for answer, label in answers:
    result = multidim_judge(question, answer)
    scores = result.get('scores', {})
    print(f"\n[{label}] {answer[:30]}...")
    for dim, score in scores.items():
        print(f"  {dim}: {score}/5")
    print(f"  요약: {result.get('summary', '')}")

### 6-1. 레이더 차트로 시각화

다차원 점수를 한 눈에 보려면 **레이더 차트(스파이더 차트)**가 효과적이다.  
각 꼭지점이 평가 차원, 면적이 넓을수록 다방면으로 좋은 답변.

In [ ]:
def plot_radar(scores_dict, title='multi-dim evaluation'):
    """레이더 차트로 다차원 점수 시각화"""
    categories = list(scores_dict.keys())
    values = list(scores_dict.values())
    values += values[:1]  # 원을 닫기 위해 첫 번째 값 추가
    
    # 각도 계산: 360도를 차원 개수로 나눠서 균등 배치
    angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.fill(angles, values, alpha=0.25)       # 내부 색칠
    ax.plot(angles, values, 'o-', linewidth=2) # 외곽선 + 점
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_ylim(0, 5)                          # 1~5점 스케일
    ax.set_title(title, pad=20)
    plt.show()

In [ ]:
# CSV 읽는 방법 질문에 대해 다차원 평가 + 시각화
question = "파이썬으로 CSV 파일을 읽는 방법은?"
answer = "pandas 라이브러리의 read_csv를 사용합니다. import pandas as pd; df = pd.read_csv('file.csv')"

result = multidim_judge(question, answer)
scores = result.get('scores', {})
print(scores, result.get('summary'))
plot_radar(scores)

## 7. LLM 평가의 편향(Bias) 탐지

LLM Judge는 편리하지만 다음 3가지 편향이 있다:

| 편향 | 설명 | 비유 |
|------|------|------|
| **위치 편향** | 앞에 있는 답변에 후하게 점수 | 시험지 첫 번째 답안을 후하게 채점하는 선생님 |
| **길이 편향** | 긴 답변에 높은 점수 | 나무위키처럼 길게 쓰면 '정성스럽다'고 느끼는 경향 |
| **자기강화 편향** | 자신과 유사한 스타일에 후하게 | GPT가 GPT스러운 글에 더 높은 점수 |

**위치 편향 대응**: A/B 순서 바꾸어서 2번 평가 (어제 실습함)  
**길이 편향 대응**: 아래에서 탐지  
**자기강화 편향 대응**: 다른 모델로 교차 검증 (GPT로 평가 + Claude로 평가)

In [ ]:
def detect_length_bias(question, short_answer, long_answer):
    """길이 편향 탐지: 짧은 답변과 긴 답변의 점수 차이를 측정
    비유: 같은 내용인데 한쪽은 길게 쓴 것. 점수 차이가 나면 편향 있음."""
    short_result = pointwise_judge(question, short_answer)
    long_result = pointwise_judge(question, long_answer)
    
    short_score = short_result.get('score')
    long_score = long_result.get('score')
    
    bias = long_score - short_score  # 양수면 긴 답변에 점수를 더 줬다는 뜻
    
    return {
        "short_score" : short_score,
        "long_score" : long_score,
        "bias" : bias,
        "short_len" : len(short_answer),
        "long_len" : len(long_answer),
        "has_bias" : bias > 0
    }

In [ ]:
# 길이 편향 테스트: 같은 내용이지만 길이가 다른 답변
question = "파이썬 GIL이란?"
short = "GIL은 한 번에 하나의 스레드만 파이썬 바이트코드를 실행하도록 제한하는 뮤텍스입니다."
long = "GIL(Global Interpreter Lock)은 CPython 인터프리터에서 사용되는 메커니즘입니다. GIL은 한 번에 하나의 스레드만 파이썬 바이트코드를 실행하도록 제한하는 뮤텍스(mutual exclusion lock)입니다. 이는 메모리 관리의 thread-safety를 위해 도입되었으나, 멀티스레드 프로그램의 병렬 실행을 제한하는 단점이 있습니다. 이를 우회하기 위해 multiprocessing 모듈을 사용하거나, I/O 바운드 작업에서는 asyncio를 활용할 수 있습니다."

result = detect_length_bias(question, short, long)
print(f"Short({result['short_len']}글자): {result['short_score']}점")
print(f"Long({result['long_len']}글자): {result['long_score']}점")
print(f"Bias: {result['bias']} ({'\ud3b8\ud5a5 \uc788\uc74c' if result['has_bias'] else '\ud3b8\ud5a5 \uc5c6\uc74c'})")

## 8. 앱상블 Judge -- 여러 평가 방법을 합치기

한 명의 심사위원보다 여러 명의 심사위원이 더 공정하다.  
Pointwise, CoT, Few-shot 점수를 **평균/절삭평균/가중평균**으로 합치는 것.

| 합산 방식 | 설명 | 비유 |
|----------|------|------|
| 평균 | 모든 점수를 더해서 나눔 | 보통 성적 산출 |
| 절삭평균 | 최대/최소 제외 후 평균 | 올림픽 심사와 같은 방식 |
| 가중평균 | 똑똑한 모델에 더 큰 가중치 | 경험 많은 심사위원 의견에 가중치 |

In [ ]:
def ensemble_judge(question, answer, judge_fns, weight=None):
    """여러 평가 함수의 점수를 합산하는 앱상블 Judge
    
    judge_fns: [("이름", 함수), ...] 형태의 리스트
    weight: 각 함수의 가중치 (None이면 동일 가중치)"""
    scores = []
    for name, fn in judge_fns:
        result = fn(question, answer)
        score = result.get('score')
        scores.append(score)
        print(f"  {name}: {score}점")
    
    # 1) 단순 평균
    mean_score = np.mean(scores)
    # 2) 중앙값
    median_score = np.median(scores)
    
    # 3) 절삭평균: 최대/최소 제외 (3개 이상일 때만 의미 있음)
    if len(scores) >= 3:
        trimmed_mean = np.mean(sorted(scores)[1:-1])
    else:
        trimmed_mean = mean_score
    
    # 4) 가중평균: weight 제공 시
    if weight:
        weighted_mean = np.average(scores, weights=weight)  # weights= 키워드 인수로 전달해야 함!
    else:
        weighted_mean = mean_score
    
    return {
        'scores': scores,
        'mean_score': mean_score,
        'median_score': median_score,
        'trimmed_mean': trimmed_mean,
        'weighted_mean': weighted_mean
    }

In [ ]:
# 3가지 Judge를 앱상블
judge_fns = [
    ("pointwise", pointwise_judge),
    ("CoT", cot_judge),
    ("Few-shot", fewshot_judge)
]

question = "API 게이트웨이란?"
answer = "API 게이트웨이는 클라이언트와 백엔드 서비스 사이의 중간 계층으로, 인증, 라우팅, 속도 제한 등을 처리합니다."

# weight=[1,2,3]: Few-shot에 가장 큰 가중치
result = ensemble_judge(question, answer, judge_fns, weight=[1, 2, 3])
print(f"\n--- 앱상블 결과 ---")
print(f"평균: {result['mean_score']:.1f}")
print(f"중앙값: {result['median_score']:.1f}")
print(f"절삭평균: {result['trimmed_mean']:.1f}")
print(f"가중평균: {result['weighted_mean']:.1f}")

## 9. 재현성 & 분별력 테스트

LLM Judge를 신뢰할 수 있는지 검증하는 2가지 테스트:

| 테스트 | 질문 | 측정 방법 | 기준 |
|--------|------|----------|------|
| **재현성** | 같은 입력으로 5번 돌리면 같은 점수? | 변동계수(CV) = std/mean | CV < 0.1이면 우수 |
| **분별력** | 좋은/나쁜 답변을 구분하는가? | 평균 점수 차이(gap) | gap > 1.5이면 우수 |

비유:
- 재현성: 체중계에 5번 올라갔을 때 매번 같은 수치가 나오는지
- 분별력: 체중계가 50kg과 80kg을 확실히 구분하는지

In [ ]:
def test_reproducibility(question, answer, n_trials=5):
    """재현성 테스트: 같은 입력으로 n번 평가해서 점수 일관성 확인"""
    scores = []
    for i in range(n_trials):
        result = pointwise_judge(question, answer)
        score = result.get('score')
        scores.append(score)
    
    mean = np.mean(scores)
    std = np.std(scores)
    # 변동계수(CV): 표준편차/평균. 작을수록 재현성이 좋다
    cv = std / mean if mean > 0 else float('inf')
    
    return {'scores': scores, 'mean': mean, 'std': std, 'cv': cv}

In [ ]:
question = "REST API란?"
answer = "REST는 HTTP 프로토콜을 사용하여 리소스를 CRUD 방식으로 관리하는 아키텍처 스타일입니다."

result = test_reproducibility(question, answer, n_trials=5)
print(f"점수들: {result['scores']}")
print(f"평균: {result['mean']:.2f}, 표준편차: {result['std']:.2f}")
print(f"변동계수(CV): {result['cv']:.3f} {'(OK: < 0.1)' if result['cv'] < 0.1 else '(재현성 부족)'}") 

In [ ]:
def test_discriminability(question, good_answer, bad_answer, n_trials=3):
    """분별력 테스트: 좋은 답변과 나쁜 답변의 점수 차이 확인"""
    good_scores = []
    bad_scores = []
    
    for _ in range(n_trials):
        good_r = pointwise_judge(question, good_answer)
        bad_r = pointwise_judge(question, bad_answer)
        good_scores.append(good_r.get('score'))
        bad_scores.append(bad_r.get('score'))
    
    gap = np.mean(good_scores) - np.mean(bad_scores)
    
    return {
        'good_mean': np.mean(good_scores),
        'bad_mean': np.mean(bad_scores),
        'gap': gap,
        'discriminable': gap > 1.5  # gap > 1.5이면 분별력 있음
    }

In [ ]:
result = test_discriminability(
    "데이터베이스 인덱스란?",
    "데이터베이스 인덱스는 테이블 검색 속도를 높이기 위한 자료구조입니다. B-Tree, Hash 등의 구조를 사용하며, SELECT 쿼리 성능을 크게 향상시킵니다.",
    "데이터베이스에 있는 것입니다."
)
print(f"좋은 답변 평균: {result['good_mean']:.1f}점")
print(f"나쁜 답변 평균: {result['bad_mean']:.1f}점")
print(f"차이(gap): {result['gap']:.1f} ({'\ubd84\ubcc4\ub825 OK' if result['discriminable'] else '\ubd84\ubcc4\ub825 \ubd80\uc871'})")

## 10. Criteria Evaluation -- 기준 기반 평가 프레임워크

위에서 배운 모든 것을 하나의 **클래스**로 정리해서 실제 프로젝트에 재사용할 수 있게 만든다.

좋은 평가 기준의 조건:
- **관찰 가능**: 예시가 포함되어 있는지 확인 가능
- **독립적**: 다른 기준과 걹치지 않음
- **측정 가능**: 1~5점으로 객관적 측정 가능
- **목적 부합**: 평가 목적에 맞는 기준

In [ ]:
class EvaluationCriterion:
    """평가 기준 객체 -- 기준을 카드처럼 정의해서 재사용"""
    def __init__(self, name, description, scale=(1,5)):
        self.name = name
        self.description = description
        self.scale = scale
    
    def __repr__(self):
        # print() 하면 자동으로 이 형식으로 출력됨
        return f"Criterion({self.name}: {self.description}, {self.scale[0]}-{self.scale[1]})"

In [ ]:
# 표준 평가 기준 4개 정의
STANDARD_CRITERIA = [
    EvaluationCriterion("accuracy",     "사실적으로 정확한 정보만 포함하는가"),
    EvaluationCriterion("relevance",    "질문의 의도와 범위에 적합한 답변인가"),
    EvaluationCriterion("completeness", "핵심 포인트를 빠짐없이 다루는가"),
    EvaluationCriterion("coherence",    "논리적 흐름과 구조가 일관적인가"),
]

# 출력 테스트
print(EvaluationCriterion("coherence", "논리적 흐름과 구조가 일관적인가"))

In [ ]:
def criteria_evaluate(question, answer, criterion):
    """하나의 평가 기준으로 답변을 점수화"""
    prompt = f"""당신은 전문 평가자입니다.
    아래 기준에 따라 답변을 {criterion.scale[0]}-{criterion.scale[1]} 스케일로 평가하세요.
    
    질문 : {question}
    답변 : {answer}
    
    평가 기준 [{criterion.name}] : {criterion.description}
    
    JSON으로 답하세요:
    {{"score" : {criterion.scale[0]}-{criterion.scale[1]}}}"""
    
    result = llm.invoke(prompt).content
    cleaned = result.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1].replace("json", "")
    return json.loads(cleaned)

In [ ]:
# 4개 기준으로 순회하며 평가
question = "도커(Docker)의 장점을 설명해주세요"
answer = "도커는 컨테이너 기술로 애플리케이션을 격리된 환경에서 실행합니다. 가볍고 빠르며, 환경 일관성을 보장합니다."

for criterion in STANDARD_CRITERIA:
    result = criteria_evaluate(question, answer, criterion)
    print(f"  {criterion.name}: {result.get('score')}/5")

## 정리

### Score Filtering
| 전략 | 언제 쓰나 |
|------|----------|
| Fixed Threshold | 절대적 품질 기준이 있을 때 |
| Dynamic Threshold | 데이터 분포가 매번 다를 때 |
| Score Gap | 점수에 자연스러운 군집이 있을 때 |
| Adaptive | 모르겠을 때 (자동 선택) |

### LLM Judge 심화
| 방법 | 핵심 |
|------|------|
| 다차원 평가 | 여러 측면에서 독립 점수화 + 레이더 차트 시각화 |
| 편향 탐지 | 길이/위치/자기강화 편향을 정량적으로 측정 |
| 앱상블 | 여러 Judge의 점수를 평균/절삭/가중으로 합산 |
| 재현성/분별력 | CV < 0.1 / gap > 1.5 기준으로 Judge 신뢰도 검증 |
| Criteria | 클래스로 평가기준 정의 -> 실무 프레임워크 |